# Introduction

In [9]:
from sympy import symbols, Eq, exp, log, sqrt, solveset, S, sin, cos
from collections import deque


def get_complexity_expression(expr):
    """
    Compute the complexity of a SymPy expression efficiently by counting the number of unique nodes 
    in its expression tree using an optimized traversal.

    Args:
        expr (sympy.Expr): The symbolic expression.

    Returns:
        int: Complexity score (total number of unique nodes in the expression tree).

    Example:
        >>> from sympy import symbols
        >>> a, x = symbols('a x')
        >>> expr = (a*x + 2) / (x + 1)
        >>> get_complexity_expression(expr)
        7
    """

    if expr == 0:
        return 1  # Complexity of constant zero is 1

    visited = set()
    stack = deque([expr])  # Use deque for fast pop/push operations
    complexity = 0

    while stack:
        node = stack.pop()

        if node in visited:
            continue  # Avoid redundant processing
        visited.add(node)

        complexity += 1  # Count this node

        # Add sub-expressions (children) efficiently
        if node.is_Atom:
            continue  # Skip atomic symbols and numbers
        stack.extend(node.args)  # Push children

    return complexity

# symbols
a, b, c, x, y = symbols('a b c x y')
# original equation
e1 = a*exp(x) + b*exp(-x) + c

e2 = e1.subs(exp(x),y)

get_complexity_expression(e1), get_complexity_expression(e2)

(11, 9)

In [16]:
import sympy as sp

# symbols
a, b, c, x, y = sp.symbols('a b c x y')

# your helper
def subs_x_with_f(lhs_rhs, f, xsym=x):
    if isinstance(lhs_rhs, tuple):
        L, R = lhs_rhs
        L = sp.sympify(L); R = sp.sympify(R)
        return (sp.simplify(L.subs(xsym, f)), sp.simplify(R.subs(xsym, f)))
    return sp.simplify(sp.sympify(lhs_rhs).subs(xsym, f))

# example equations and change-of-variables:
# 1) Quadratic: y = x + b/(2a)  ->  x = y - b/(2a)
# 2) Exponential: y = exp(x)     ->  x = log(y)
eqns = [
    a*x**2 + b*x + c,
    a*sp.exp(x) + b*sp.exp(-x) + c
]
subs = [
    y - b/(2*a),     # x -> y - b/(2a)
    sp.log(y)        # x -> log(y)
]

# your complexity function assumed to be available
# from your earlier code; using the same name:
# get_complexity_expression(expr)

for e1, f in zip(eqns, subs):
    e2 = subs_x_with_f(e1, f, xsym=x)     # apply x -> f(y,params)
    c1 = get_complexity_expression(e1)
    c2 = get_complexity_expression(e2)
    print(f'{sp.simplify(e1)} complexity = {c1}')
    print(f'{sp.simplify(e2)} complexity = {c2}\n')


NameError: name 'get_complexity_expression' is not defined

In [3]:
from sympy import symbols, simplify, exp, log, Wild

def pi_cov_general(main_eqn):
    """
    Return a change-of-variables sub-expression to assign into `x`
    (i.e., the env will do `x := <return value>`), choosing among:
      • Quadratic  ax^2+bx+c            →  x ↦ x - B/(2A)  (complete the square)
      • Cubic      ax^3+bx^2+cx+d       →  x ↦ x - B/(3A)  (depress the cubic)
      • Quartic    ax^4+bx^3+cx^2+dx+e  →  x ↦ x - B/(4A)  (depress the quartic)
      • Exponential a e^{k x} + b e^{-k x} + c →  x ↦ log(x)/k
        (so the equation becomes a*x + b/x + c = 0 in the new variable)

    Returns None if no supported pattern is detected.
    """
    s = simplify(main_eqn)

    # ----- Polynomial cases (deg = 2,3,4): use shift to kill the next-highest term
    poly = s.as_poly(x)
    if poly is not None:
        deg = poly.degree()
        if deg in (2, 3, 4):
            coeffs = poly.all_coeffs()
            A = coeffs[0]          # leading
            B = coeffs[1]          # next (x^(deg-1)) coefficient
            if A != 0:
                if deg == 2:       # quadratic: kill linear term
                    return x - B/(2*A)
                elif deg == 3:     # cubic: kill x^2 term
                    return x - B/(3*A)
                elif deg == 4:     # quartic: kill x^3 term
                    return x - B/(4*A)

    # ----- Exponential case: a*exp(k*x) + b*exp(-k*x) + c
    # Use Wilds to detect the structure and read off k
    aW = Wild('aW', exclude=[x])
    bW = Wild('bW', exclude=[x])
    cW = Wild('cW', exclude=[x])
    kW = Wild('kW', exclude=[x])

    # try exact 3-term form (up to simplification)
    m = s.match(aW*exp(kW*x) + bW*exp(-kW*x) + cW)
    if m and kW in m and m[kW] != 0:
        k = m[kW]
        # Substitute x := (1/k)*log(x). Then exp(k*x) → x and exp(-k*x) → 1/x.
        return log(x)/k

    # No supported pattern detected
    return None

from sympy import symbols, expand, Poly, exp, log, simplify

# reuse your function pi_cov_general(...)

# fresh symbols for coefficients
A2,B2,C2 = symbols('A2 B2 C2')
A3,B3,C3,D3 = symbols('A3 B3 C3 D3')
A4,B4,C4,D4,E4 = symbols('A4 B4 C4 D4 E4')
ka, kb, kc, kk = symbols('ka kb kc kk', nonzero=True)

# 1) Quadratic: x ↦ x - B/(2A) kills linear term
s2 = A2*x**2 + B2*x + C2
sub2 = pi_cov_general(s2)
assert sub2 == x - B2/(2*A2)
s2_new = expand(s2.subs(x, sub2))
assert Poly(s2_new, x).coeff_monomial(x) == 0

# 2) Cubic: x ↦ x - B/(3A) kills x^2 term
s3 = A3*x**3 + B3*x**2 + C3*x + D3
sub3 = pi_cov_general(s3)
assert sub3 == x - B3/(3*A3)
s3_new = expand(s3.subs(x, sub3))
assert Poly(s3_new, x).coeff_monomial(x**2) == 0

# 3) Quartic: x ↦ x - B/(4A) kills x^3 term
s4 = A4*x**4 + B4*x**3 + C4*x**2 + D4*x + E4
sub4 = pi_cov_general(s4)
assert sub4 == x - B4/(4*A4)
s4_new = expand(s4.subs(x, sub4))
assert Poly(s4_new, x).coeff_monomial(x**3) == 0

# 4) Exponential template: a*e^{k x} + b*e^{-k x} + c → set y=e^{k x}
sexp = ka*exp(kk*x) + kb*exp(-kk*x) + kc
sube = pi_cov_general(sexp)
assert simplify(sube - log(x)/kk) == 0
sexp_new = simplify(sexp.subs(x, sube))
assert simplify(sexp_new - (ka*x + kb/x + kc)) == 0  # now rational in x

# 5) Negative case: no recognized pattern -> None
assert pi_cov_general(x**5 + 1) is None

print("All CoV policy tests passed ✔️")

All CoV policy tests passed ✔️


In [25]:
import sympy as sp
from sympy import symbols, simplify

# your helper
def subs_x_with_f(lhs_rhs, f, xsym=sp.Symbol('x')):
    if isinstance(lhs_rhs, tuple):
        L, R = lhs_rhs
        L = sp.sympify(L); R = sp.sympify(R)
        return (sp.simplify(L.subs(xsym, f)), sp.simplify(R.subs(xsym, f)))
    return sp.simplify(sp.sympify(lhs_rhs).subs(xsym, f))

# assume pi_cov_general(...) is already defined (handles quadratic/cubic/quartic/exp cases)

a, b, c, d, e, x, y = symbols('a b c d e x y')

# --- Cubic example (eq1): choose c = b^2/(3a) so the depressed cubic has no y-term ---
eq1 = a*x**3 + b*x**2 + (b**2/(3*a))*x + d

# --- Quartic example (eq2): choose
#     c = 3*b^2/(8*a) and d = b**3/(16*a**2)
# so that after the Tschirnhaus shift x -> y - b/(4a) we get a*y**4 + (e - b**4/(256*a**3))
c_star = 3*b**2/(8*a)
d_star = b**3/(16*a**2)
eq2 = a*x**4 + b*x**3 + c_star*x**2 + d_star*x + e

# Apply the CoV suggested by the policy
sub1 = pi_cov_general(eq1)  # should return x - b/(3*a) for cubic
sub2 = pi_cov_general(eq2)  # should return x - b/(4*a) for quartic

# Substitute x -> sub and simplify
eq1_dep = subs_x_with_f(eq1, sub1, xsym=x)
eq2_dep = subs_x_with_f(eq2, sub2, xsym=x)

print("sub1 =", sub1)
print("depressed cubic =", sp.expand(eq1_dep))

print("sub2 =", sub2)
print("depressed quartic =", sp.expand(eq2_dep))


sub1 = x - b/(3*a)
depressed cubic = a*x**3 + d - b**3/(27*a**2)
sub2 = x - b/(4*a)
depressed quartic = a*x**4 + e - b**4/(256*a**3)


: 

### Things here

In [2]:
for a in env.actions:
    print(a)

NameError: name 'env' is not defined

In [ ]:
p